# 003 · Comments and Docstrings

> **Syntax › Basics** · Python 3.14

Python has exactly one comment syntax, `#`, plus **docstrings** — string literals that document modules, classes and functions and, unlike C# XML doc comments, are available **at runtime**.

**In this notebook you will learn:**

- line comments with `#`, and why there is no `/* ... */` block comment
- what docstrings are and how they compare to C#'s `///` XML doc comments
- how to read documentation at runtime with `help()` and `__doc__`
- the common docstring styles (Google, NumPy, reStructuredText) and the PEP 257 conventions

## C# mental model

| C# | Python |
|---|---|
| `// line comment` | `# line comment` |
| `/* block comment */` | no equivalent — use several `#` lines |
| `/// <summary>...</summary>` XML doc comment | `"""Docstring."""` as the first statement of the body |
| doc comments compiled to an `.xml` file, read by IntelliSense | docstrings stored on the object in `__doc__`, read by `help()`, IDEs and doc generators |
| `#region` / `#pragma` | no equivalent (editors fold by indentation; linters use `# noqa`-style comments) |

## Line comments with `#`

Everything from `#` to the end of the line is ignored — exactly like `//` in C#. Comments can stand on their own line or follow code (PEP 8 asks for at least two spaces before an inline `#`).

```csharp
// C#
// Calculate the discounted price
var price = basePrice * 0.9m;   // 10% off
```

In [ ]:
# Calculate the discounted price
base_price = 50.0
price = base_price * 0.9   # 10% off

print(price)

A `#` inside a string literal is just a character, not a comment:

In [ ]:
hashtag = "#python"   # only this part is a comment
print(hashtag)

## No block comments

There is no `/* ... */` in Python. Multi-line comments are simply several `#` lines — every editor has a "toggle comment" shortcut (`Ctrl+/` / `Cmd+/`) that adds or removes them for a selection.

```csharp
// C#
/*
 * Retry policy:
 * up to three attempts, doubling the delay each time.
 */
```

In [ ]:
# Retry policy:
# up to three attempts, doubling the delay each time.
delays = [1, 2, 4]
print(delays)

⚠️ **Gotcha:** you will sometimes see a bare triple-quoted string used as a "block comment". It works only because an expression statement whose value is unused is evaluated and thrown away — it is **not** a comment. It can't appear in every position a comment can, and when it is the first statement of a function it silently becomes that function's docstring. Prefer `#` lines.

In [ ]:
"""
This string is evaluated and discarded.
It is not a real comment.
"""
print("the string above did nothing visible")

## Docstrings

A **docstring** is a string literal that appears as the **first statement** of a module, class, function or method. Python stores it on the object as `__doc__`, so documentation travels with the code at runtime.

```csharp
// C#
/// <summary>Returns the area of a rectangle.</summary>
/// <param name="width">Width in metres.</param>
/// <param name="height">Height in metres.</param>
/// <returns>The area in square metres.</returns>
public static double Area(double width, double height) => width * height;
```

Note the placement difference: a C# doc comment goes **above** the member; a Python docstring goes **inside**, directly under the `def` line.

In [ ]:
def area(width, height):
    """Return the area of a rectangle."""
    return width * height

print(area(3, 4))
print(area.__doc__)   # the docstring is a real attribute on the function object

By convention docstrings always use triple double quotes `"""..."""`, even for one-liners, so they can grow to multiple lines without changing quote style.

Classes and modules have docstrings too:

In [ ]:
class Point:
    """A point in 2D space."""

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance_to_origin(self):
        """Return the Euclidean distance from (0, 0)."""
        return (self.x ** 2 + self.y ** 2) ** 0.5

print(Point.__doc__)
print(Point.distance_to_origin.__doc__)

import math
print(math.__doc__)   # a module's docstring is the first statement in its file

A comment is **not** a docstring — comments are discarded by the compiler, so a function documented only with `#` has no `__doc__`:

In [ ]:
def no_docs():
    # this is just a comment
    return 42

print(no_docs.__doc__)   # None

## Reading docs at runtime: `help()`

`help()` formats an object's docstring together with its signature — the runtime equivalent of hovering over a member to see its IntelliSense tooltip. It works on your own code and on every built-in.

In [ ]:
help(area)

In [ ]:
help(len)

In Jupyter you can also type `area?` in a cell (or press `Shift+Tab` inside the parentheses) to see the same information. The `inspect.getdoc()` function returns a docstring with its indentation cleaned up, which is handy when processing docs programmatically.

## Multi-line docstrings and PEP 257

PEP 257 defines the docstring conventions:

- the first line is a **short summary** ending in a period, written as a command ("Return …", not "Returns …")
- for longer docstrings: summary line, a **blank line**, then the details
- the closing `"""` goes on its own line for multi-line docstrings

In [ ]:
import inspect

def transfer(amount, source, target):
    """Move an amount between two accounts.

    The source balance is checked before anything is changed, so a
    failed transfer leaves both accounts untouched.
    """
    ...

print(inspect.getdoc(transfer))

## Docstring styles

PEP 257 says *where* docstrings go, but not how to describe parameters. Unlike C#'s single XML schema (`<param>`, `<returns>`, `<exception>`), Python has three popular styles. Pick one per project — tools like Sphinx (with the `napoleon` extension) and IDEs understand all of them.

**Google style** — the most readable, very common in application code:

In [ ]:
def divide(dividend, divisor):
    """Divide two numbers.

    Args:
        dividend: The number to be divided.
        divisor: The number to divide by. Must not be zero.

    Returns:
        The quotient as a float.

    Raises:
        ZeroDivisionError: If ``divisor`` is zero.
    """
    return dividend / divisor

print(divide(10, 4))

**NumPy style** — popular in scientific libraries (NumPy, pandas, SciPy):

```python
def divide(dividend, divisor):
    """Divide two numbers.

    Parameters
    ----------
    dividend : float
        The number to be divided.
    divisor : float
        The number to divide by. Must not be zero.

    Returns
    -------
    float
        The quotient.
    """
```

**reStructuredText (Sphinx) style** — the classic Sphinx field-list format, closest in spirit to XML tags:

```python
def divide(dividend, divisor):
    """Divide two numbers.

    :param dividend: The number to be divided.
    :param divisor: The number to divide by. Must not be zero.
    :returns: The quotient.
    :raises ZeroDivisionError: If divisor is zero.
    """
```

| C# XML tag | Google | NumPy | reST |
|---|---|---|---|
| `<summary>` | first line | first line | first line |
| `<param name="x">` | `Args:` | `Parameters` | `:param x:` |
| `<returns>` | `Returns:` | `Returns` | `:returns:` |
| `<exception cref="...">` | `Raises:` | `Raises` | `:raises ...:` |

💡 With type hints in the signature *(see Type Hints › 001 · Type Annotations)*, you usually don't need to repeat parameter types in the docstring — just describe what the values mean.

## Summary

| Concept | C# | Python |
|---|---|---|
| Line comment | `// text` | `# text` |
| Block comment | `/* ... */` | several `#` lines |
| API documentation | `/// <summary>` above the member | `"""docstring"""` as first statement inside |
| Doc available at runtime | no (separate `.xml` file) | yes — `obj.__doc__` |
| View documentation | IntelliSense tooltip | `help(obj)`, `obj?` in Jupyter |
| Parameter docs | `<param>`, `<returns>`, `<exception>` | Google / NumPy / reST sections |
| Conventions | Microsoft docs guidelines | PEP 257 |

## Exercises

1. Write a function `celsius_to_fahrenheit(celsius)` with a one-line docstring, then print its `__doc__`.
2. Add a Google-style docstring with `Args:` and `Returns:` sections to a function `full_name(first, last)` and view it with `help()`.
3. Without running it, predict what `print(mystery.__doc__)` outputs:
   ```python
   def mystery():
       x = 1
       """Is this the docstring?"""
       return x
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def celsius_to_fahrenheit(celsius):
    """Convert a temperature from Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32

print(celsius_to_fahrenheit.__doc__)

# 2
def full_name(first, last):
    """Build a display name.

    Args:
        first: The given name.
        last: The family name.

    Returns:
        The name formatted as "First Last".
    """
    return f"{first.title()} {last.title()}"

help(full_name)

# 3 — prints None. A docstring must be the FIRST statement of the body;
#     here the string is just an expression that is evaluated and discarded.
```

</details>